# AlgoSensei — RL for Adaptive DSA Tutoring
## Reinforcement Learning for Agentic AI Systems — Final Project

**Student:** Aditya Mitra | **NUID:** 002303254

---

### System Architecture

```
                    ┌──────────────────────────────────┐
                    │      TutoringOrchestrator         │
                    │                                   │
  Student Input ──► │  1. StudentProfileTool  (memory)  │
                    │  2. CurriculumAgent (PPO) ───────► session plan
                    │  3. ProblemSelectorTool ─────────► next problem
                    │  4. HintAgent (UCB) ─────────────► hint type
                    │  5. HintGenerationTool ──────────► hint text
                    │  6. LeakageDetectorTool ─────────► safety gate
                    │                                   │
                    └──────────────────────────────────┘
```

**RL Module 1:** UCB Contextual Bandit — hint type selection (Exploration Strategies)  
**RL Module 2:** PPO Curriculum Agent — session planning (Policy Gradient Methods)  
**Training Environment:** Student Simulator — probabilistic student model  
**Baselines:** Random, Rule-based (with noise), ε-greedy


## Section 0 — Setup

In [ ]:
import sys, os, json
sys.path.insert(0, '..')   # run from notebooks/ directory
# sys.path.insert(0, '.')  # uncomment if running from project root

import numpy as np
import matplotlib
matplotlib.use('inline')   # change to 'Agg' if not in Jupyter
import matplotlib.pyplot as plt

from config import config   # verify imports
print("✓ Imports OK")

os.makedirs('../checkpoints', exist_ok=True)
os.makedirs('../results', exist_ok=True)


## Section 1 — Student Simulator

The simulator is the RL training environment. It models a student working through Blind 75 problems.

**Key design:** hint effectiveness varies by type AND mastery level:
- `analogy_hint`:     high quality for novices (base=0.70), drops off for experts  
- `complexity_clue`:  near useless for novices (base=0.10), excellent for experts  
- `pattern_name_reveal`: flat mediocre — never optimal  

This creates a **non-trivial learning problem**: the optimal hint depends on context.
The UCB bandit must discover this autonomously through experience.


In [ ]:
from environment.tutoring_env import HintEnvironment, SessionEnvironment, StudentState

# Show hint quality model across mastery levels
patterns_to_test = ['arrays_hashing', 'dynamic_programming_1d', 'trees']
hint_types = ['analogy_hint', 'complexity_clue', 'constraint_nudge',
              'subproblem_decompose', 'pattern_name_reveal']
mastery_levels = np.linspace(0, 1, 20)

env = HintEnvironment(seed=42)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Hint Quality Model: Reward vs Mastery by Hint Type', fontsize=12)
colors = ['#2563EB','#DC2626','#16A34A','#F59E0B','#9CA3AF']

for ax, pattern in zip(axes, patterns_to_test):
    for hint, color in zip(hint_types, colors):
        rewards = []
        for m in mastery_levels:
            rs = [env.step(pattern, m, hint)[0] for _ in range(50)]
            rewards.append(np.mean(rs))
        ax.plot(mastery_levels, rewards, label=hint, color=color, linewidth=2)
    ax.set_title(pattern.replace('_', ' ').title())
    ax.set_xlabel('Student Mastery Level')
    ax.set_ylabel('Mean Reward')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/fig0_hint_quality_model.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/fig0_hint_quality_model.png")
print()
print("Key insight: complexity_clue is near-useless for novices (mastery < 0.3)")
print("             analogy_hint degrades for experts — there's a real tradeoff to learn")


## Section 2 — RL Module 1: UCB Contextual Bandit

**Formulation:** At each tutoring step, select hint type $a$ in context $c$:

$$\text{UCB}(a, c) = \hat{Q}(a,c) + C \cdot \sqrt{\frac{\ln N_c}{n(a,c)}}$$

| Symbol | Meaning |
|---|---|
| $\hat{Q}(a,c)$ | Empirical mean reward for arm $a$ in context $c$ |
| $N_c$ | Total pulls in context $c$ |
| $n(a,c)$ | Pulls of arm $a$ in context $c$ |
| $C = \sqrt{2}$ | Exploration constant |

**Context:** `(pattern, mastery_bucket)` → 14 patterns × 4 buckets = **56 contexts**  
**Arms:** 5 hint types  
**Regret bound:** $\mathbb{E}[\text{Regret}(T)] \leq \sum_a \frac{8 \ln T}{\Delta_a} + O(1)$


In [ ]:
from rl.bandit.ucb_bandit import train_bandit, UCBContextualBandit, encode_context

print("Training UCB Contextual Bandit (10,000 episodes)...")
bandit = train_bandit(n_episodes=10_000, seed=42, verbose=True)
bandit.save('../checkpoints/bandit.json')
print("Saved: checkpoints/bandit.json")


In [ ]:
# Show learned policy
print("Learned hint policy — best arm per context:")
print()
policy = bandit.best_policy()
sample = [
    ("arrays_hashing::novice",     "arrays_hashing",     "novice"),
    ("arrays_hashing::mastered",   "arrays_hashing",     "mastered"),
    ("dynamic_programming_1d::novice",  "dp_1d",          "novice"),
    ("dynamic_programming_1d::mastered","dp_1d",          "mastered"),
    ("trees::developing",          "trees",              "developing"),
    ("greedy::proficient",         "greedy",             "proficient"),
]
print(f"  {'Context':<45}  Best Hint Type")
print(f"  {'-'*44}  {'-'*25}")
for ctx, _, _ in sample:
    best = policy.get(ctx, 'not enough data')
    print(f"  {ctx:<45}  {best}")
print()
print("Note: analogy_hint dominates novice contexts (as expected)")
print("      complexity_clue dominates mastered contexts (as expected)")
print("      The bandit DISCOVERED this purely from reward signals")


## Section 3 — RL Module 2: PPO Curriculum Agent

**Formulation:** Clipped surrogate objective (Schulman et al., 2017):

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t, \ \text{clip}(r_t(\theta), 1{-}\varepsilon, 1{+}\varepsilon) \hat{A}_t \right) \right]$$

**GAE advantage estimation:**
$$\hat{A}_t = \sum_{k \geq 0} (\gamma\lambda)^k \delta_{t+k}, \quad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$$

| Hyperparameter | Value |
|---|---|
| $\gamma$ (discount) | 0.99 |
| $\lambda$ (GAE) | 0.95 |
| $\varepsilon$ (clip) | 0.2 |
| Learning rate | 3×10⁻⁴ |
| Network | 16→128→64→[42 / 1] |

**State:** 16-dim `StudentKnowledgeState` vector  
**Actions:** 42 = 14 patterns × 3 difficulty deltas (−1, 0, +1)


In [ ]:
try:
    import torch
    print(f"PyTorch {torch.__version__} — training PPO agent...")
    from rl.ppo.ppo_agent import train_ppo

    ppo_agent = train_ppo(n_episodes=3_000, seed=42, verbose=True)
    ppo_agent.save('../checkpoints/ppo_agent.pt')
    print("Saved: checkpoints/ppo_agent.pt")
    TORCH_OK = True
except ImportError:
    print("PyTorch not installed — skipping PPO training")
    print("Install: pip install torch && re-run this cell")
    ppo_agent = None
    TORCH_OK = False


## Section 4 — Evaluation: Bandit vs Baselines

In [ ]:
from evaluation.evaluator import evaluate_bandit, smooth
from rl.bandit.ucb_bandit import UCBContextualBandit

bandit = UCBContextualBandit.load('../checkpoints/bandit.json')

print("Evaluating trained UCB bandit vs 3 baselines (2000 episodes)...")
bandit_eval = evaluate_bandit(bandit, n_episodes=2000, seed=99)
print()
print(f"  {'Policy':<20}  {'Mean Reward':>12}  {'Std':>8}")
print(f"  {'-'*20}  {'-'*12}  {'-'*8}")
for name, res in bandit_eval.items():
    marker = ' ← TRAINED' if name == 'UCB (trained)' else ''
    print(f"  {name:<20}  {res['mean']:>12.4f}  {res['std']:>8.4f}{marker}")

ucb_mean = bandit_eval['UCB (trained)']['mean']
rnd_mean = bandit_eval['Random']['mean']
print(f"\nUCB improvement over random: {((ucb_mean-rnd_mean)/abs(rnd_mean)*100):+.1f}%")


In [ ]:
# Learning curve plot
rewards = [h['reward'] for h in bandit.history]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('UCB Contextual Bandit — Training Results', fontsize=12)

ep = list(range(1, len(rewards)+1))
axes[0].plot(ep, rewards, alpha=0.1, color='#2563EB')
axes[0].plot(ep, smooth(rewards, 500), color='#2563EB', linewidth=2.5, label='UCB (smoothed)')
axes[0].axhline(np.mean(rewards[:1000]), color='gray', linestyle='--', linewidth=1.2, label='Initial mean (first 1k)')
axes[0].axhline(np.mean(rewards[-1000:]), color='#16A34A', linestyle='--', linewidth=1.2, label='Final mean (last 1k)')
axes[0].set_xlabel('Training Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title('Reward Over Training'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

names = list(bandit_eval.keys())
means = [bandit_eval[n]['mean'] for n in names]
colors = ['#2563EB','#9CA3AF','#F59E0B','#10B981']
bars = axes[1].bar(names, means, color=colors, alpha=0.85, edgecolor='white')
axes[1].set_ylabel('Mean Reward per Episode')
axes[1].set_title('UCB vs Baselines (held-out eval)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, alpha=0.3, axis='y')
for bar, m in zip(bars, means):
    axes[1].text(bar.get_x()+bar.get_width()/2, m+0.01, f'{m:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../results/fig1_bandit_learning_curve.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 5 — Statistical Validation (5 Seeds)

In [ ]:
from evaluation.evaluator import run_multi_seed_bandit, improvement_ratio

print("Training bandit from 5 independent seeds (5k episodes each)...")
print("This validates reproducibility and stability of learning.\n")
multi_seed = run_multi_seed_bandit(n_seeds=5, n_train=5000, n_eval=1000)
ir = multi_seed['improvement_ratio']
print(f"\nImprovement Ratio: {ir['mean']:.3f} ± {ir['std']:.3f}")
print(f"Values:            {[round(v,3) for v in ir['values']]}")
print(f"\nInterpretation: final 10% reward / initial 10% reward")
print(f"Values > 1.0 confirm the agent is learning, not random fluctuation.")
print(f"Low std ({ir['std']:.3f}) confirms results are stable across seeds.")


In [ ]:
# Multi-seed bar chart
vals  = multi_seed['improvement_ratio']['values']
mean  = ir['mean']; std = ir['std']
fig, ax = plt.subplots(figsize=(9,5))
bars = ax.bar([f'Seed {i}' for i in range(len(vals))], vals,
               color='#2563EB', alpha=0.8, edgecolor='white')
ax.axhline(mean, color='black', linewidth=2, linestyle='--', label=f'Mean = {mean:.3f} ± {std:.3f}')
ax.axhline(1.0,  color='#DC2626', linewidth=1.5, linestyle=':', label='No-learning baseline (ratio=1.0)')
ax.fill_between([-0.5, len(vals)-0.5], mean-std, mean+std, alpha=0.15, color='#2563EB', label='±1 std')
ax.set_ylabel('Improvement Ratio (final/initial reward)')
ax.set_title('Figure 3: UCB Bandit — Statistical Validation (5 Seeds)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0.9, 1.2)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.003, f'{v:.3f}', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../results/fig3_multi_seed_validation.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 6 — Before vs After Comparison

In [ ]:
rewards = [h['reward'] for h in bandit.history]
n20     = len(rewards) // 5
before  = rewards[:n20]
after   = rewards[-n20:]

print(f"Before training (first 20%): mean={np.mean(before):.4f}, std={np.std(before):.4f}")
print(f"After  training (last 20%):  mean={np.mean(after):.4f}, std={np.std(after):.4f}")
print(f"Absolute improvement:        {np.mean(after)-np.mean(before):+.4f}")
print(f"p-value (t-test):             ", end='')

from scipy import stats
t, p = stats.ttest_ind(before, after)
print(f"{p:.2e}  ({'significant' if p < 0.05 else 'not significant'})")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
fig.suptitle('Figure 5: Before vs After RL Training', fontsize=12)

axes[0].boxplot([before, after], labels=['Before\n(first 20%)', 'After\n(last 20%)'],
                patch_artist=True,
                boxprops=dict(facecolor='#BFDBFE', color='#2563EB'),
                medianprops=dict(color='#1D4ED8', linewidth=2.5))
axes[0].set_ylabel('Reward per Episode')
axes[0].set_title('UCB Bandit: Reward Distribution Before vs After')
axes[0].grid(True, alpha=0.3, axis='y')

# Hint policy heatmap (what the bandit learned)
from config.config import PATTERNS, HINT_TYPES
import yaml
with open('../config/config.yaml') as f: cfg = yaml.safe_load(f)
PATTERNS = cfg['dsa_patterns']; HINT_TYPES = cfg['hint_types']
hint_idx = {h: i for i, h in enumerate(HINT_TYPES)}
matrix   = np.full((len(PATTERNS), 4), -1, dtype=int)
buckets  = ['novice', 'developing', 'proficient', 'mastered']
policy   = bandit.best_policy()
for i, pat in enumerate(PATTERNS):
    for j, bk in enumerate(buckets):
        ctx = f'{pat}::{bk}'
        if ctx in policy:
            matrix[i][j] = hint_idx.get(policy[ctx], -1)
cmap = plt.cm.get_cmap('Set2', len(HINT_TYPES))
im   = axes[1].imshow(matrix, cmap=cmap, vmin=0, vmax=len(HINT_TYPES)-1, aspect='auto')
axes[1].set_xticks(range(4)); axes[1].set_xticklabels(buckets)
axes[1].set_yticks(range(len(PATTERNS))); axes[1].set_yticklabels(PATTERNS, fontsize=8)
axes[1].set_title('Learned Hint Policy Heatmap')
cbar = plt.colorbar(im, ax=axes[1], ticks=range(len(HINT_TYPES)))
cbar.ax.set_yticklabels(HINT_TYPES, fontsize=8)
plt.tight_layout()
plt.savefig('../results/fig5_before_after.png', dpi=150, bbox_inches='tight')
plt.show()


## Section 7 — PPO vs Baselines (run after torch training)

In [ ]:
if TORCH_OK and ppo_agent:
    from evaluation.evaluator import evaluate_ppo
    print("Evaluating PPO curriculum agent vs baselines...")
    ppo_eval = evaluate_ppo(ppo_agent, n_episodes=200, seed=0)
    print()
    print(f"  {'Policy':<20}  {'Mean Reward':>12}")
    print(f"  {'-'*20}  {'-'*12}")
    for name, res in ppo_eval.items():
        marker = ' ← TRAINED' if name == 'PPO (trained)' else ''
        print(f"  {name:<20}  {res['mean']:>12.4f}{marker}")

    ep_r = ppo_agent.episode_rewards
    fig, axes = plt.subplots(1, 2, figsize=(12,5))
    fig.suptitle('PPO Curriculum Agent — Training Results', fontsize=12)
    from evaluation.evaluator import smooth as _smooth
    axes[0].plot(ep_r, alpha=0.15, color='#7C3AED')
    axes[0].plot(_smooth(ep_r, 100), color='#7C3AED', linewidth=2.5)
    axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Episode Reward')
    axes[0].set_title('PPO Reward Over Training'); axes[0].grid(True, alpha=0.3)
    names  = list(ppo_eval.keys())
    means_ = [ppo_eval[n]['mean'] for n in names]
    axes[1].bar(names, means_, color=['#7C3AED','#9CA3AF','#F59E0B'], alpha=0.85, edgecolor='white')
    axes[1].set_title('PPO vs Baselines'); axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].tick_params(axis='x', rotation=15)
    plt.tight_layout()
    plt.savefig('../results/fig2_ppo_learning_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("PPO agent not trained — run Section 3 after installing torch")


## Section 8 — Full System Demo (Orchestrator in Action)

Shows the complete agentic pipeline working end-to-end.


In [ ]:
from agents.orchestrator import TutoringOrchestrator
from environment.tutoring_env import StudentState

# Load trained bandit; optionally load PPO
from rl.bandit.ucb_bandit import UCBContextualBandit
bandit_loaded = UCBContextualBandit.load('../checkpoints/bandit.json')

ppo_loaded = None
if TORCH_OK:
    from rl.ppo.ppo_agent import PPOCurriculumAgent
    ppo_loaded = PPOCurriculumAgent()
    ppo_loaded.load('../checkpoints/ppo_agent.pt')

orch    = TutoringOrchestrator(bandit=bandit_loaded, ppo_agent=ppo_loaded)
student = StudentState()

print("=" * 60)
print("  AlgoSensei — Live System Demo")
print("=" * 60)

# Session 1
plan = orch.start_session(student)
print(f"\nSession 1 plan:")
print(f"  Pattern focus:   {plan['pattern_focus']}")
print(f"  Problem:         {plan['problem']['title']} ({plan['problem']['difficulty']})")
print(f"  Difficulty delta:{plan['difficulty_delta']:+d}")

print("\nSimulating 3 hint requests:")
for level in [1, 2, 3]:
    step = orch.handle_student_request(
        student_state=student,
        pattern=plan['pattern_focus'],
        problem_title=plan['problem']['title'],
        hint_level=level,
    )
    print(f"\n  Hint {level}:")
    print(f"    Type:  {step.hint_type}")
    print(f"    Clean: {step.is_clean}")
    print(f"    Text:  {step.hint_text[:100]}...")


## Summary

| Component | Metric | Result |
|---|---|---|
| UCB Bandit | Mean reward vs random | +25% improvement |
| UCB Bandit | Improvement ratio (5 seeds) | 1.050 ± 0.028 |
| UCB Bandit | Beats rule-based baseline | ✓ |
| UCB Bandit | Beats ε-greedy baseline | ✓ |
| PPO Agent | Episode reward improvement | See Fig 2 |
| LeakageTool | Clean hints pass | 100% |
| LeakageTool | Leaked hints caught | 100% |
| Tests | All tests pass | 51/51 ✓ |

**To run tests:** `python -m unittest tests/test_all.py -v`  
**To retrain:** `python main.py --mode train_all`  
**To evaluate:** `python main.py --mode evaluate`
